# THEMol NSP QM Reference Instances

Downloads the **THEMol Hessian** subset from HuggingFace
(`ByteDance-Seed/THEMol`) and extracts per-instance geometry records for
N-, S-, and P-containing molecules matched to OpenFF 2.3.0 parameters.

Output is one `{param_id}_themol_instances.json` file per target parameter,
in the same `InstanceRecord` format expected by the ff_param_splitter scripts
(`analyze_parameter.py`, `validate_hierarchy.py`).

## Workflow

1. Download the CSV index (~30 MB) listing all 3.1 M Hessian molecules.
2. Filter rows whose SMILES contain N, S, or P atoms (fast string match).
3. Group matched molecules by H5 file; download only those files.
4. For each molecule: assign OpenFF parameters, measure bond/angle geometry
   from the QM-optimised coordinates (Å, degrees), build `InstanceRecord`.
5. Accumulate records per `param_id` and save as JSON.

THEMol uses **B3LYP-D3(BJ)/DZVP** level of theory — the same as the NSP
QCArchive datasets used in the existing ff_param_splitter workflow.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION — edit this cell, then run it before anything else
# ---------------------------------------------------------------------------
import os
from pathlib import Path

# ------------------------------------------------------------------
# HuggingFace cache location — SET THIS BEFORE anything else.
# huggingface_hub reads HF_HOME at import time; setting it afterward
# has no effect on the already-imported module.
# Leave as None to use the default ~/.cache/huggingface.
HF_CACHE_DIR: str | None = None  # e.g. '/scratch/your_username/hf_cache'

# HuggingFace auth token. If the env lookup below returns NOT FOUND,
# paste your token string directly:  HF_TOKEN = "hf_xxxxxxxxxxxx"
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')

# Apply to env before importing huggingface_hub so module reads correct paths.
if HF_CACHE_DIR:
    os.environ['HF_HOME'] = HF_CACHE_DIR
    os.environ['HUGGINGFACE_HUB_CACHE'] = HF_CACHE_DIR  # legacy name
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
# ------------------------------------------------------------------

import huggingface_hub

print(f'HF_TOKEN : {"FOUND" if HF_TOKEN else "NOT FOUND — set it above manually"}')
print(f'HF cache : {HF_CACHE_DIR or "~/.cache/huggingface (default)"}')

if HF_TOKEN:
    huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'Authenticated ({len(HF_TOKEN)} char token, huggingface_hub {huggingface_hub.__version__}).')
else:
    print('WARNING: proceeding unauthenticated — downloads will be rate-limited.')

# ------------------------------------------------------------------

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate repo root from {start}')

REPO_ROOT = _find_repo_root()

# OpenFF force field for parameter assignment
FF_NAME = 'openff-2.3.0.offxml'

# Root output directory for this notebook
OUTPUT_DIR = REPO_ROOT / 'examples' / '10_themol_qm_reference' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Intermediate parquet — stores ALL bond+angle instances for ALL NSP molecules.
INTERMEDIATE_PARQUET = OUTPUT_DIR / 'themol_nsp_all_instances.parquet'

# Directory for per-parameter JSON files (consumed by ff_param_splitter scripts)
INSTANCES_DIR = (
    REPO_ROOT
    / 'examples'
    / '07_single_model_benchmark'
    / 'ff_param_splitter'
    / 'instances_themol'
)
INSTANCES_DIR.mkdir(parents=True, exist_ok=True)

# HuggingFace repo
HF_REPO = 'ByteDance-Seed/THEMol'

# Worker processes for parallel molecule processing. Lower if sharing the node.
N_WORKERS: int = os.cpu_count()

# Limit to N H5 files for a quick test run (None = process all ~50 files)
MAX_H5_FILES: int | None = None

print(f'Worker processes  : {N_WORKERS}')
print(f'Repo root         : {REPO_ROOT}')
print(f'Intermediate file : {INTERMEDIATE_PARQUET}')
print(f'Instances dir     : {INSTANCES_DIR}')

In [2]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.inchi import MolToInchi, MolToInchiKey
from tqdm.auto import tqdm

from huggingface_hub import hf_hub_download

from openff.toolkit import ForceField, Molecule, Topology

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from mlip_optimizer.analysis.ff_param_splitter.extract import InstanceRecord, save_instances

print('Imports OK')

Imports OK


## Step 1 — Download the Hessian CSV index

The CSV is ~30 MB and lists all 3.1 M molecules with their mapped SMILES and
which H5 file they live in.  It is downloaded once and cached by
`huggingface_hub`.

In [ ]:
csv_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='Hessian/hessian_dataset.csv',
    repo_type='dataset',
    cache_dir=HF_CACHE_DIR,
    token=HF_TOKEN,
)
print(f'CSV at: {csv_path}')

index_df = pd.read_csv(csv_path)
print(f'Total molecules in Hessian subset: {len(index_df):,}')
print(f'Columns: {index_df.columns.tolist()}')
print(f'Unique H5 files: {index_df["h5_file"].nunique()}')
index_df.head(3)

## Step 2 — Filter for N / S / P-containing molecules

A fast string-level pre-filter retains SMILES that contain `N`, `S`, or `P`
characters (uppercase = aliphatic; aromatic atoms `n`/`s`/`p` are also caught
by adding their lowercase forms).  No RDKit parsing needed at this stage.

In [4]:
smiles_col = 'mapped_nonisomeric_smiles'

nsp_mask = index_df[smiles_col].str.contains('[NSPnsp]', regex=True)
nsp_df = index_df[nsp_mask].copy()

print(f'NSP molecules : {len(nsp_df):,}  ({len(nsp_df)/len(index_df):.1%} of total)')

h5_files_needed = nsp_df['h5_file'].unique()
print(f'H5 files to download: {len(h5_files_needed)}')

if MAX_H5_FILES is not None:
    h5_files_needed = h5_files_needed[:MAX_H5_FILES]
    nsp_df = nsp_df[nsp_df['h5_file'].isin(h5_files_needed)]
    print(f'(Limited to {MAX_H5_FILES} H5 files → {len(nsp_df):,} molecules)')

# Group molecule rows by H5 file for efficient access
h5_groups = {h5: grp for h5, grp in nsp_df.groupby('h5_file')}
print(f'Ready to process {len(h5_groups)} H5 files.')

NSP molecules : 2,753,137  (88.7% of total)
H5 files to download: 50
Ready to process 50 H5 files.


## Step 3 — Load force field (main process) and verify `_worker.py`

In [ ]:
import sys
from pathlib import Path
from openff.toolkit import ForceField

_NB_DIR = str(REPO_ROOT / 'examples' / '10_themol_qm_reference')
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)
import _worker

# Quick sanity check: run the worker init + one dummy call in the main process
ff = ForceField(FF_NAME)
print(f'ForceField loaded: {FF_NAME}')
print(f'_worker.py found : {Path(_worker.__file__).resolve()}')
print('Step 3 OK — ready for parallel processing.')

## Step 4 — Download H5 files and extract all instances

Each H5 file is downloaded once (cached by `huggingface_hub`) and processed
in parallel using **`ProcessPoolExecutor` with `spawn`**.  Each worker process
gets a fresh Python interpreter and loads its own `ForceField` once via the
initializer — this avoids the C-extension init conflicts that crash
`ThreadPoolExecutor` on cluster nodes.

Only `coords` is read per molecule; the large `hessian` arrays are skipped.

**Disk space:** ~50 H5 files × ~50–200 MB each ≈ up to ~5 GB download.
Set `MAX_H5_FILES = 1` above for a quick smoke-test.

In [ ]:
import multiprocessing as mp
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed

import pyarrow as pa
import pyarrow.parquet as pq

# Make _worker.py importable by spawned subprocesses
_NB_DIR = str(REPO_ROOT / 'examples' / '10_themol_qm_reference')
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)
import _worker  # noqa: E402 — must be after sys.path update

SCHEMA = pa.schema([
    ('mol_idx',    pa.int32()),
    ('inchi_key',  pa.string()),
    ('smiles',     pa.string()),
    ('cmiles',     pa.string()),
    ('param_type', pa.string()),
    ('atom_key',   pa.string()),
    ('param_id',   pa.string()),
    ('smirks',     pa.string()),
    ('qm_value',   pa.float64()),
])

_SRC_PATH = str(REPO_ROOT / 'src')
_SPAWN_CTX = mp.get_context('spawn')

print(f'Using {N_WORKERS} worker processes (spawn).')

mol_counter = 0
n_skipped   = 0
n_processed = 0
n_instances = 0

pq_writer: pq.ParquetWriter | None = None

try:
    for h5_name, mol_rows in tqdm(h5_groups.items(), desc='H5 files'):
        try:
            h5_local = hf_hub_download(
                repo_id=HF_REPO,
                filename=f'Hessian/{h5_name}',
                repo_type='dataset',
                cache_dir=HF_CACHE_DIR,
                token=HF_TOKEN,
            )
        except Exception as e:
            print(f'  WARNING: could not download {h5_name}: {e}')
            continue

        uuid_smiles = dict(zip(mol_rows['uuid'], mol_rows[smiles_col]))

        # Phase 1: sequential H5 read
        mol_data: list[tuple[str, list, int]] = []
        with h5py.File(h5_local, 'r') as hf:
            for uuid, cmiles in uuid_smiles.items():
                if uuid not in hf:
                    n_skipped += 1
                    continue
                coords = hf[uuid]['coords'][:].tolist()  # list for pickle across processes
                mol_data.append((cmiles, coords, mol_counter))
                mol_counter += 1

        # Phase 2: parallel processing
        batch_rows: list[dict] = []
        with ProcessPoolExecutor(
            max_workers=N_WORKERS,
            mp_context=_SPAWN_CTX,
            initializer=_worker.init,
            initargs=(FF_NAME, _SRC_PATH),
        ) as pool:
            futures = {
                pool.submit(_worker.process, cmiles, coords, mol_idx): mol_idx
                for cmiles, coords, mol_idx in mol_data
            }
            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=h5_name, leave=False):
                try:
                    batch_rows.extend(fut.result())
                except Exception:
                    pass

        n_processed += len(mol_data)

        # Phase 3: write parquet chunk
        if batch_rows:
            table = pa.Table.from_pylist(batch_rows, schema=SCHEMA)
            if pq_writer is None:
                pq_writer = pq.ParquetWriter(INTERMEDIATE_PARQUET, SCHEMA)
            pq_writer.write_table(table)
            n_instances += len(batch_rows)

        # Phase 4: delete the H5 file now that it is safely written to parquet.
        # hf_hub_download returns a symlink; resolve it to get the actual blob.
        try:
            h5_path = Path(h5_local)
            blob = h5_path.resolve()   # real file
            blob.unlink()              # frees disk space
            h5_path.unlink(missing_ok=True)  # remove dangling symlink
        except Exception as e:
            print(f'  WARNING: could not delete {h5_name} cache file: {e}')

finally:
    if pq_writer is not None:
        pq_writer.close()

print(f'\nProcessed : {n_processed:,} molecules  |  skipped: {n_skipped:,}')
print(f'Instances : {n_instances:,} rows written to {INTERMEDIATE_PARQUET.name}')
print(f'File size : {INTERMEDIATE_PARQUET.stat().st_size / 1e6:.1f} MB')

## Step 5 — Inspect the intermediate parquet

Run this cell at any time (including after the download is long finished) to
see which parameters are covered and how many instances exist for each.

In [ ]:
all_df = pd.read_parquet(INTERMEDIATE_PARQUET)
print(f'Total rows : {len(all_df):,}')
print(f'Unique mols: {all_df["mol_idx"].nunique():,}')

summary = (
    all_df
    .groupby(['param_type', 'param_id'])
    .agg(
        n_instances=('qm_value', 'count'),
        n_mols=('mol_idx', 'nunique'),
        qm_min=('qm_value', 'min'),
        qm_mean=('qm_value', 'mean'),
        qm_max=('qm_value', 'max'),
        qm_width=('qm_value', lambda x: x.max() - x.min()),
    )
    .reset_index()
    .sort_values(['param_type', 'param_id'])
)
summary

## Step 6 — Export per-parameter JSON files for ff_param_splitter

Writes one `{param_id}_themol_instances.json` for **every** parameter ID
present in the parquet — i.e. every bond and angle parameter that OpenFF 2.3.0
assigned to at least one NSP molecule.  No re-downloading required.

In [ ]:
def _parquet_row_to_instance(row: pd.Series) -> InstanceRecord:
    """Reconstruct an InstanceRecord from one parquet row."""
    atom_key = tuple(int(x) for x in row['atom_key'].split(','))
    return InstanceRecord.from_dict(
        {
            'mol_idx':   int(row['mol_idx']),
            'inchi_key': row['inchi_key'],
            'smiles':    row['smiles'],
            'cmiles':    row['cmiles'],
            'atom_key':  list(atom_key),
            'qm_values': [float(row['qm_value'])],
            'qm_mean':   float(row['qm_value']),
            'mm_values': {},
            'errors':    {},
            'param_id':  row['param_id'],
            'smirks':    row['smirks'],
        },
        reconstruct_rdmol=True,
    )


df = pd.read_parquet(INTERMEDIATE_PARQUET)

for pid, grp in tqdm(df.groupby('param_id'), desc='Exporting'):
    instances = [_parquet_row_to_instance(row) for _, row in grp.iterrows()]
    out_path  = INSTANCES_DIR / f'{pid}_themol_instances.json'
    save_instances(instances, out_path)
    vals = [r.qm_mean for r in instances]
    unit = 'Å' if pid.startswith('b') else '°'
    print(
        f'{pid:6s}: {len(instances):>7,} instances  '
        f'[{min(vals):.3f}, {max(vals):.3f}] {unit}  '
        f'-> {out_path.name}'
    )

print(f'\n{df["param_id"].nunique()} parameter files written to: {INSTANCES_DIR}')

## Using the output with ff_param_splitter

The JSON files written by Step 6 are drop-in replacements for the NSP
QCArchive instance files.

### Analyze a parameter

```bash
cd examples/07_single_model_benchmark/ff_param_splitter

python scripts/analyze_parameter.py \
    instances_themol/a20_themol_instances.json --param a20

python scripts/analyze_parameter.py \
    instances_themol/a20_themol_instances.json --param a20 --subpartition
```

### Validate a proposed hierarchy

```bash
python scripts/validate_hierarchy.py \
    instances_themol/a20_themol_instances.json \
    --param a20 \
    --inline '[
        ["[*:1]~[#7X3$(*~[#6X3,#6X2,#7X2+0]):2]~[*:3]", "a20"],
        ["[*:1]-[#7X3;!r:2]-[#7X2,#7X3:3]",              "a20f"],
        ["[#6;r6:1]-[#7X3;r6:2]-[#6;r6:3]",              "a20a"]
    ]'
```

### Combine THEMol + QCArchive instances

```python
from mlip_optimizer.analysis.ff_param_splitter.extract import load_instances, save_instances

qca  = load_instances('a20_instances.json')
them = load_instances('instances_themol/a20_themol_instances.json')

offset = max(r.mol_idx for r in qca) + 1
for r in them:
    r.mol_idx += offset

save_instances(qca + them, 'a20_combined_instances.json')
```

### Re-export a different parameter set later

Just edit `EXPORT_PARAM_IDS` in Step 6 and re-run that single cell — no
re-downloading, no reprocessing.  The parquet holds everything.

---

**Notes on THEMol vs QCArchive data:**
- Each THEMol instance has exactly **one conformer** (`qm_values` length 1);
  QCArchive instances typically have 5–10.  Both feed into `qm_mean` the same way.
- `mm_values` and `errors` are empty (no MM comparison).
- QM level: B3LYP-D3(BJ)/DZVP — identical to the NSP QCArchive sets.